# 4. Decoder-Only Transformer

Decoder-only transformers (like GPT) use masked self-attention and feed-forward networks.
This architecture powers modern language models!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


Using device: cpu


## Key Components

1. **Masked Multi-Head Attention** - Can only attend to previous positions
2. **Feed-Forward Network** - Two linear layers with activation
3. **Layer Normalization** - Normalizes activations
4. **Residual Connections** - Adds input to output (x + f(x))


In [5]:
# defining sequence_length
seq_len = 5

# creating input 
x = torch.ones(seq_len, seq_len)

# Masked attention: prevent attending to future positions
# Causal mask or known as look-ahead mask: prevent model from 'looking' into future
#   In training, the mask used to prevent the model to 'see' the answer without predicting.
#   Ensuring the model only attends word from specific range to specific range

# Create causal mask (higher triangular) by filling with ones
#   Example: x - (5,5) tensor object, the matrix shape square are split into 2 part (or 2 triangles)
#               this initialize the higher part of triangle (higher triangular - triu()) filled in 1s
#   Diagonal = 1 used to shift the diagonal up by = 1 as offset from the 'split' of diagonal
mask = torch.triu(x, diagonal=1)
# now, defining the if the mask == 1(true), then it becomes -inf
mask = mask.masked_fill(mask == 1, float('-inf'))
# if the mask == 0(false), its sets it value as 0.0
mask = mask.masked_fill(mask == 0, 0.0)

print("Causal mask (0 = allowed, -inf = masked):")
print(mask)
print("\nEach position can only attend to itself and previous positions!")


Causal mask (0 = allowed, -inf = masked):
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

Each position can only attend to itself and previous positions!


## Masked Multi-Head Attention

Same as multi-head attention, but with causal masking applied to attention scores.


In [ ]:
class MaskedMultiHeadAttention(nn.Module):
    """Masked multi-head attention for decoder"""
    
    def __init__(self, d_model, num_heads):
        # inheriting the pytorch NN libraries
        super().__init__()
        
        # ensuring the dimension_model (d_model) is divisible by number_heads (num_heads)
        #   if true, the runtime will pass and run the execution
        #   else (false), the runtime will halt and throws AssertionError Exception
        # it is to ensure the divisible d_model % num_heads == 0 for splitting across multi-heads 
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads

        # calculating dimension_key (d_k)
        #   done by floor division (// operation) by 
        #       ignoring the float value and round it off the integer value 
        self.d_k = d_model // num_heads
        
        # initializing the weights for Q(Query), K(Key), V(Value) and O(Output)
        #   in linear projection
        #   in defining the features = d_model
        # defining 'bias=False' 
        #   - stick the formula for calculating attention score (not including bias)
        #   - reducing unecessary trainable parameters
        #   - biases are needed but not in here 
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # unlike Q, K and V - initializing the weights for O(Output)
        #   in linear projection by defining the feature = d_model
        #   withouth 'bias=False', therefore bias is enabled
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x, mask=None):
        """
        Args:
            x: [batch_size, seq_len, d_model]
            mask: Optional causal mask [seq_len, seq_len]
        """
        # defining batch_size, seq_len and d_model taken from x.shape
        batch_size, seq_len, d_model = x.shape
        
        # inserting the x input into linear weights for Q, K and V 
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape for multi-head
        #   by referencing teh original shape (batch_size, seq_len, d_model)
        #       into multi-head (batch_size, seq_len, num_heads, d_k)
        #   then, transpose by swapping dimension dim1 and dim2 becomes
        #        (batch_seize, num_heads, seq_len, d_k) <-- requirement for matmul later on
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        #   Calculate attention_score (QK^T / sqrt(d_k)) (1st and 2nd step)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Show attention_score dimension
        print(f"Attention score shape: {scores.shape}")
        print()

        # Show one part of attention_scores before masking
        #   Remember: array dimension can be represent in 3-dimension
        #           the print() can be represented in 3 innermost dimension
        print("Attention score before masking (values inside tensor object as tokens):")
        print("(num_heads, seq_len, seq_len)")
        print(f"Attention_score[0,0,0] = {scores[0,0,0]}")
        print(f"Attention_score[0,0,1] = {scores[0,0,1]}")
        print()

        # Apply causal mask if provided
        # causal mask properties defined in main execution
        # copy the masking from 'mask' provided into attention_scores
        if mask is not None:
            scores = scores.masked_fill(mask == float('-inf'), float('-inf'))

        # Show one part of same attention_scores after masking
        print("Attention score after applying causal_masking (values inside tensor object as tokens):")
        print("(num_heads, seq_len, seq_len)")
        print(f"Attention_score[0,0,0] = {scores[0,0,0]}")
        print(f"Attention_score[0,0,1] = {scores[0,0,1]}")
        print()
        
        # Calculate attention weight (3rd step)
        #   by calculating ranging through softmax in innermost layer
        attention_weights = F.softmax(scores, dim=-1)
        print(f"Attention weight shape: {attention_weights.shape}")
        print()
        print("Masking also applied here after the softmax() (sets to zero(0))")
        print(f"Attention_weight[0,0,0] = {attention_weights[0,0,0]}")
        print(f"Attention_weight[0,0,1] = {attention_weights[0,0,1]}")
        print()

        # Calculate the output (4th step)
        attention_output = torch.matmul(attention_weights, V)
        
        # Concatenate heads
        # Combine the multi-heads to single Q, K and V vectors
        attention_output = attention_output.transpose(1, 2).contiguous()
        # Reshape back into original (batch_size, seq_len, d_model)
        attention_output = attention_output.view(batch_size, seq_len, d_model)
        
        # Calculate the output the projecting the attention_output as input
        #   in weight output
        output = self.W_o(attention_output)
        return output

# Test masked attention
batch_size = 2
seq_len = 10
num_heads = 8
d_model = 64

# Create causal mask
# Create causal mask that similar dimension as attention_scores (seq_len, seq_len)
# in higher higher triangle diagonal with offset = 1 (diagonal=1)
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
# set the mask properties 1 -> -inf
causal_mask = causal_mask.masked_fill(causal_mask == 1, float('-inf'))

# Creating masked multi head attention with x input
mha = MaskedMultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)

print(f"Input shape: {x.shape}")
print()
print(f"Weight for Q, K and V (bias=False): {mha.W_q}")
print(f"Weight for O (bias=True): {mha.W_o}")
print()
output = mha(x, mask=causal_mask)
print(f"Output shape: {output.shape}")
print("Masked multi-head attention with causal masking!")


Input shape: torch.Size([2, 10, 64])

Weight for Q, K and V (bias=False): Linear(in_features=64, out_features=64, bias=False)
Weight for O (bias=True): Linear(in_features=64, out_features=64, bias=True)

Attention score shape: torch.Size([2, 8, 10, 10])

Attention score before masking (values inside tensor object as tokens):
(num_heads, seq_len, seq_len)
Attention_score[0,0,0] = tensor([-0.3416, -0.0419, -0.3117,  0.5392, -0.1238,  0.2163, -0.0458, -0.0816,
        -0.1499, -0.3741], grad_fn=<SelectBackward0>)
Attention_score[0,0,1] = tensor([ 0.4874,  0.1080,  0.0774, -0.3679,  0.2291, -0.0101,  0.0265, -0.2301,
        -0.4356,  0.3302], grad_fn=<SelectBackward0>)

Attention score after applying causal_masking (values inside tensor object as tokens):
(num_heads, seq_len, seq_len)
Attention_score[0,0,0] = tensor([-0.3416,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
           -inf,    -inf], grad_fn=<SelectBackward0>)
Attention_score[0,0,1] = tensor([0.4874, 0.1080,

In [4]:
class FeedForward(nn.Module):
    """Feed-forward network"""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Expand: d_model -> d_ff, then contract: d_ff -> d_model
        return self.linear2(self.relu(self.linear1(x)))

# Test feed-forward
d_model = 64
d_ff = 256  # Typically 4x d_model
seq_len = 10
batch_size = 2

ff = FeedForward(d_model, d_ff)
x = torch.randn(batch_size, seq_len, d_model)

output = ff(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Expanded to {d_ff} dimensions, then back to {d_model}")


Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Expanded to 256 dimensions, then back to 64


## Transformer Decoder Block

Combines masked attention, feed-forward, layer norm, and residual connections.


In [5]:
class TransformerDecoderBlock(nn.Module):
    """Single decoder block"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MaskedMultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Self-attention with residual and norm
        attn_output = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual and norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

# Test decoder block
d_model = 64
num_heads = 8
d_ff = 256
seq_len = 10
batch_size = 2

causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
causal_mask = causal_mask.masked_fill(causal_mask == 1, float('-inf'))

block = TransformerDecoderBlock(d_model, num_heads, d_ff)
x = torch.randn(batch_size, seq_len, d_model)

output = block(x, mask=causal_mask)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print("Decoder block with masked attention, feed-forward, and residuals!")


Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Decoder block with masked attention, feed-forward, and residuals!


## Complete Decoder-Only Transformer

Stack multiple decoder blocks to create a full transformer model.


In [6]:
class DecoderOnlyTransformer(nn.Module):
    """Decoder-only transformer (GPT-style)"""
    
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Token and positional embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        
        # Stack of decoder blocks
        self.blocks = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Args:
            x: Token indices [batch_size, seq_len]
        Returns:
            logits: [batch_size, seq_len, vocab_size]
        """
        batch_size, seq_len = x.shape
        
        # Create causal mask
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        
        # Token embeddings
        token_emb = self.token_embedding(x)  # [batch_size, seq_len, d_model]
        
        # Positional embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        pos_emb = self.position_embedding(positions)  # [1, seq_len, d_model]
        
        # Combine embeddings
        x = self.dropout(token_emb + pos_emb)
        
        # Pass through decoder blocks
        for block in self.blocks:
            x = block(x, mask=mask)
        
        # Final layer norm and projection
        x = self.ln_f(x)
        logits = self.head(x)
        
        return logits

# Test complete transformer
vocab_size = 1000
d_model = 128
num_heads = 8
num_layers = 4
d_ff = 512
max_seq_len = 256

model = DecoderOnlyTransformer(
    vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len
)

# Example input: token indices
batch_size = 2
seq_len = 20
x = torch.randint(0, vocab_size, (batch_size, seq_len))

logits = model(x)

print(f"Input shape: {x.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\nDecoder-only transformer complete!")


Input shape: torch.Size([2, 20])
Output logits shape: torch.Size([2, 20, 1000])
Model parameters: 1,080,576

Decoder-only transformer complete!
